[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/main/notebooks/patterns/langgraph_patterns.ipynb)

# Seven matched patterns with LangGraph

**Task.** Answer *Which interventions reduce household food waste, and under what conditions?* while comparing seven graph-based orchestration patterns.

The bounded catalogue has three tutorial records: smaller plates, meal planning, and an information-only campaign. Each record has a source ID, title, topics and evidence text. Every framework notebook uses these same records and deterministic model decisions, so the comparison isolates orchestration.

**Read this notebook as:** setup → shared state/contracts → native graphs for the seven patterns → matched evaluation. Runtime is under one minute on CPU; no key or model download is required.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.9", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import json
import operator
import subprocess
import sys
from pathlib import Path
from typing import Annotated, ClassVar, TypedDict

from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, Send
from pydantic import BaseModel, ConfigDict, Field

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))

from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.schemas import (  # noqa: E402
    CriticDecision,
    Message,
    MessageRole,
    PlanDecision,
    RouteDecision,
    ToolDefinition,
)

fixture_data = json.loads(
    (ROOT / "data/research_assistant/pattern_mock_scenarios.json").read_text()
)
catalogue = fixture_data["catalogue"]
TASK = "Which interventions reduce household food waste, and under what conditions?"

## What comes from where

- **LangGraph:** `StateGraph`, `START`/`END`, `Command`, `Send`, `ToolNode` and `tools_condition` provide nodes, transitions, routing, fan-out and tool execution.
- **LangChain Core:** `AIMessage` and `@tool` provide the message/tool format consumed by `ToolNode`.
- **Pydantic:** validates structured decisions and final answers.
- **Repository:** the deterministic client, versioned fixtures, shared route/plan/critic schemas, messages and tool contract.
- **This notebook:** evidence schemas, graph state types, catalogue search, model adapter functions and acceptance checks.

The graphs—not a shared Python dispatcher—own sequencing, conditional routing, parallel fan-out, tool execution and bounded loops. Catalogue records remain read-only data.

In [ ]:
# --- Notebook-defined contracts, state and adapters ---
class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Claim(StrictModel):
    schema_id: ClassVar[str] = "tutorial.claim.v1"
    claim: str
    source_id: str
    supported: bool


class Answer(StrictModel):
    schema_id: ClassVar[str] = "tutorial.pattern_answer.v1"
    answer: str
    source_ids: tuple[str, ...]


class ParallelDecision(StrictModel):
    schema_id: ClassVar[str] = "tutorial.parallel.v1"
    queries: tuple[str, ...] = Field(min_length=2, max_length=3)
    aggregation: str = Field(pattern="^validated_union$")


class WorkerAssignment(StrictModel):
    worker: str = Field(pattern="^(intervention_reviewer|planning_reviewer)$")
    query: str


class OrchestrationDecision(StrictModel):
    schema_id: ClassVar[str] = "tutorial.orchestration.v1"
    assignments: tuple[WorkerAssignment, ...] = Field(min_length=1, max_length=2)


OrchestrationDecision.model_rebuild(_types_namespace={"WorkerAssignment": WorkerAssignment})


class FanoutState(TypedDict, total=False):
    decision: dict
    query: str
    batches: Annotated[list[list[dict]], operator.add]
    source_ids: list[str]
    trace: list[str]
    stop: str


class RouteState(TypedDict, total=False):
    route: dict
    results: list[dict]
    trace: list[str]
    stop: str


class WorkerState(TypedDict, total=False):
    decision: dict
    assignment: dict
    worker_results: Annotated[list[dict], operator.add]
    answer: dict
    trace: list[str]
    stop: str


class ReactState(MessagesState, total=False):
    answer: dict
    trace: list[str]
    steps: int
    stop: str


# The repository test runner compiles notebook cells with postponed annotations.
# Resolve reducer-bearing state types explicitly for LangGraph's runtime inspection.
FanoutState.__annotations__.update(
    batches=Annotated[list[list[dict]], operator.add],
)
WorkerState.__annotations__.update(
    worker_results=Annotated[list[dict], operator.add],
)


def model_for(name: str) -> DeterministicMockClient:
    fixture = ScriptedScenarioFixture.model_validate(
        {
            "fixture_version": fixture_data["fixture_version"],
            "scenario": name,
            "provenance": fixture_data["provenance"],
            "responses": fixture_data["scenarios"][name],
        }
    )
    return DeterministicMockClient(fixture)


def user(text: str) -> Message:
    return Message(role=MessageRole.USER, content=text)


def search(query: str, limit: int = 3) -> list[dict]:
    terms = set(query.casefold().split())
    return [
        record for record in catalogue if terms & set(" ".join(record["topics"]).casefold().split())
    ][:limit]


@tool
def search_catalogue(query: str, max_results: int) -> str:
    """Search the bounded food-waste catalogue."""
    return json.dumps(search(query, max_results))


search_contract = ToolDefinition(
    name="search_catalogue",
    description="Search the bounded food-waste catalogue.",
    parameters={
        "type": "object",
        "properties": {"query": {"type": "string"}, "max_results": {"type": "integer"}},
        "required": ["query", "max_results"],
    },
)


async def propose(client, schema, instruction):
    response = await client.generate([user(instruction)], response_schema=schema)
    return schema.model_validate(response.structured_output)

## 1–2. Prompt chaining and routing

Prompt chaining is a fixed two-node graph. Routing uses LangGraph's `Command` to update state and jump to the selected specialist; no external dispatcher selects the next function.

In [ ]:
# --- 1. Prompt chaining: graph edges own extract → synthesise ---
chain_model = model_for("prompt_chaining")


async def extract_node(state):
    claim = await propose(
        chain_model, Claim, "Extract one supported claim with source_id food-waste-001."
    )
    return {"claim": claim.model_dump(), "trace": ["extract"]}


async def synthesise_node(state):
    answer = await propose(chain_model, Answer, f"Synthesise only {state['claim']}.")
    return {
        "answer": answer.model_dump(),
        "trace": [*state["trace"], "synthesise"],
        "stop": "criteria_met",
    }


chain_builder = StateGraph(dict)
chain_builder.add_node("extract", extract_node)
chain_builder.add_node("synthesise", synthesise_node)
chain_builder.add_edge(START, "extract")
chain_builder.add_edge("extract", "synthesise")
chain_builder.add_edge("synthesise", END)
chain_state = await chain_builder.compile().ainvoke({})


# --- 2. Routing: Command performs the native state update + goto ---
route_model = model_for("routing")


async def classify_node(state):
    route = await propose(route_model, RouteDecision, f"Route this task: {TASK}")
    destination = route.route if route.route in {"research", "clarify"} else "clarify"
    return Command(
        update={"route": route.model_dump(), "trace": ["route_validated"]},
        goto=destination,
    )


def research_node(state):
    return {
        "results": search("meal planning portion"),
        "trace": [*state["trace"], "research"],
        "stop": "criteria_met",
    }


def clarify_node(state):
    return {"results": [], "trace": [*state["trace"], "clarify"], "stop": "safe_fallback"}


routing_builder = StateGraph(RouteState)
routing_builder.add_node("classify", classify_node, destinations=("research", "clarify"))
routing_builder.add_node("research", research_node)
routing_builder.add_node("clarify", clarify_node)
routing_builder.add_edge(START, "classify")
routing_builder.add_edge("research", END)
routing_builder.add_edge("clarify", END)
route_state = await routing_builder.compile().ainvoke({})

## 3–5. Parallelisation, ReAct and planner–executor

`Send` creates one graph branch per query and joins reducer-backed results. ReAct converts the fixture's tool proposal into an `AIMessage`; `ToolNode` executes the registered function tool. The planner–executor is a two-node dependency graph.

In [ ]:
# --- 3. Parallelisation: Send fans out; the list reducer joins ---
parallel_model = model_for("parallelisation")


async def plan_parallel(state):
    decision = await propose(
        parallel_model, ParallelDecision, "Propose two independent evidence searches."
    )
    return {"decision": decision.model_dump(), "batches": [], "trace": ["planned"]}


def fan_out_queries(state):
    return [Send("search_branch", {"query": query}) for query in state["decision"]["queries"]]


def search_branch(state):
    return {"batches": [search(state["query"], 2)]}


def aggregate_branches(state):
    records = {record["source_id"]: record for batch in state["batches"] for record in batch}
    return {
        "source_ids": sorted(records),
        "trace": [*state.get("trace", ["planned"]), "send", "validated_union"],
        "stop": "criteria_met",
    }


parallel_builder = StateGraph(FanoutState)
parallel_builder.add_node("plan", plan_parallel)
parallel_builder.add_node("search_branch", search_branch)
parallel_builder.add_node("aggregate", aggregate_branches)
parallel_builder.add_edge(START, "plan")
parallel_builder.add_conditional_edges("plan", fan_out_queries, ["search_branch"])
parallel_builder.add_edge("search_branch", "aggregate")
parallel_builder.add_edge("aggregate", END)
parallel_state = await parallel_builder.compile().ainvoke({})


# --- 4. ReAct: ToolNode executes the native function-tool boundary ---
react_model = model_for("react")


async def decide_tool(state):
    response = await react_model.generate(
        [user("Choose the next bounded tool call.")], tools=[search_contract]
    )
    if len(response.tool_calls) != 1:
        return {"trace": ["invalid_tool"], "steps": 0, "stop": "invalid_tool"}
    call = response.tool_calls[0]
    message = AIMessage(
        content="",
        tool_calls=[
            {
                "name": call.name,
                "args": call.arguments,
                "id": call.call_id,
                "type": "tool_call",
            }
        ],
    )
    return {"messages": [message], "trace": ["tool_proposed"], "steps": 1}


async def finish_from_observation(state):
    answer = await propose(
        react_model,
        Answer,
        f"Answer only from the tool observation: {state['messages'][-1].content}",
    )
    return {
        "answer": answer.model_dump(),
        "trace": [*state["trace"], "tool_executed", "finish"],
        "stop": "criteria_met",
    }


react_builder = StateGraph(ReactState)
react_builder.add_node("decide", decide_tool)
react_builder.add_node("tools", ToolNode([search_catalogue]))
react_builder.add_node("finish", finish_from_observation)
react_builder.add_edge(START, "decide")
react_builder.add_conditional_edges("decide", tools_condition, {"tools": "tools", END: END})
react_builder.add_edge("tools", "finish")
react_builder.add_edge("finish", END)
react_state = await react_builder.compile().ainvoke({"messages": [], "trace": []})


# --- 5. Planner-executor: separate graph nodes enforce dependencies ---
plan_model = model_for("planner_executor")


async def plan_node(state):
    plan = await propose(plan_model, PlanDecision, f"Create a bounded plan for: {TASK}")
    return {"plan": plan.model_dump(), "trace": ["plan_validated"]}


def execute_plan_node(state):
    steps = state["plan"]["steps"]
    valid = len(steps) == 3 and all(
        tuple(steps[index].get("depends_on", ())) == (steps[index - 1]["step_id"],)
        for index in (1, 2)
    )
    return {
        "trace": [*state["trace"], "executed" if valid else "dependency_stop"],
        "stop": "criteria_met" if valid else "invalid_plan",
    }


planner_builder = StateGraph(dict)
planner_builder.add_node("plan", plan_node)
planner_builder.add_node("execute", execute_plan_node)
planner_builder.add_edge(START, "plan")
planner_builder.add_edge("plan", "execute")
planner_builder.add_edge("execute", END)
plan_state = await planner_builder.compile().ainvoke({})

## 6–7. Critic–reviser and orchestrator–worker

A conditional edge permits at most one revision. The orchestrator uses `Send` to create specialist worker branches and a reducer to join their evidence before synthesis.

In [ ]:
# --- 6. Critic-reviser: conditional edge + explicit revision budget ---
critic_model = model_for("critic_reviser")


async def draft_node(state):
    draft = await propose(critic_model, Answer, "Draft an answer.")
    return {"draft": draft.model_dump(), "revisions": 0, "trace": ["draft"]}


async def critique_node(state):
    decision = await propose(
        critic_model, CriticDecision, f"Check support and conditions: {state['draft']}"
    )
    return {
        **state,
        "decision": decision.model_dump(),
        "trace": [*state["trace"], "critic"],
    }


def revision_route(state):
    return "revise" if not state["decision"]["accepted"] and state["revisions"] < 1 else END


async def revise_node(state):
    answer = await propose(
        critic_model, Answer, f"Revise once for these issues: {state['decision']['issues']}"
    )
    return {
        "draft": answer.model_dump(),
        "revisions": state["revisions"] + 1,
        "trace": [*state["trace"], "revision"],
        "stop": "criteria_met",
    }


critic_builder = StateGraph(dict)
critic_builder.add_node("draft", draft_node)
critic_builder.add_node("critic", critique_node)
critic_builder.add_node("revise", revise_node)
critic_builder.add_edge(START, "draft")
critic_builder.add_edge("draft", "critic")
critic_builder.add_conditional_edges("critic", revision_route, {"revise": "revise", END: END})
critic_builder.add_edge("revise", END)
critic_state = await critic_builder.compile().ainvoke({}, {"recursion_limit": 6})


# --- 7. Orchestrator-worker: Send creates validated worker branches ---
orchestrator_model = model_for("orchestrator_worker")


async def assign_workers(state):
    decision = await propose(
        orchestrator_model, OrchestrationDecision, "Assign two bounded reviewers."
    )
    return {
        "decision": decision.model_dump(),
        "worker_results": [],
        "trace": ["assignments_validated"],
    }


def dispatch_workers(state):
    return [
        Send("worker", {"assignment": assignment})
        for assignment in state["decision"]["assignments"]
    ]


def worker_node(state):
    assignment = state["assignment"]
    return {
        "worker_results": [
            {
                "worker": assignment["worker"],
                "records": search(assignment["query"], 2),
            }
        ]
    }


async def synthesise_workers(state):
    answer = await propose(
        orchestrator_model, Answer, f"Synthesise worker evidence: {state['worker_results']}"
    )
    return {
        "answer": answer.model_dump(),
        "trace": [
            *state.get("trace", ["assignments_validated"]),
            "send_workers",
            "synthesis_validated",
        ],
        "stop": "criteria_met",
    }


orchestrator_builder = StateGraph(WorkerState)
orchestrator_builder.add_node("assign", assign_workers)
orchestrator_builder.add_node("worker", worker_node)
orchestrator_builder.add_node("synthesise", synthesise_workers)
orchestrator_builder.add_edge(START, "assign")
orchestrator_builder.add_conditional_edges("assign", dispatch_workers, ["worker"])
orchestrator_builder.add_edge("worker", "synthesise")
orchestrator_builder.add_edge("synthesise", END)
orchestrator_state = await orchestrator_builder.compile().ainvoke({})

## Evaluation summary

Each result must terminate explicitly and satisfy the same observable acceptance check as the other framework notebooks.

Limitations: graph topology makes control flow inspectable but cannot guarantee semantic correctness; concurrent branches can duplicate work; and the deterministic fixture tests orchestration rather than real-model quality.

In [ ]:
pattern_evaluations = {
    "prompt_chaining": chain_state["stop"] == "criteria_met"
    and chain_state["trace"] == ["extract", "synthesise"],
    "routing": route_state["stop"] == "criteria_met" and len(route_state["results"]) == 2,
    "parallelisation": parallel_state["source_ids"] == ["food-waste-001", "food-waste-002"],
    "react": react_state["stop"] == "criteria_met"
    and react_state["trace"] == ["tool_proposed", "tool_executed", "finish"],
    "planner_executor": plan_state["stop"] == "criteria_met",
    "critic_reviser": critic_state["stop"] == "criteria_met" and critic_state["revisions"] == 1,
    "orchestrator_worker": orchestrator_state["stop"] == "criteria_met"
    and len(orchestrator_state["worker_results"]) == 2,
}
pattern_limitations = {
    "prompt_chaining": "early errors propagate",
    "routing": "valid routes can still be semantically wrong",
    "parallelisation": "branches can duplicate work",
    "react": "tool loops require strict budgets",
    "planner_executor": "invalid dependencies stop execution",
    "critic_reviser": "one revision may be insufficient",
    "orchestrator_worker": "worker findings can conflict",
}
assert set(pattern_limitations) == set(pattern_evaluations)
assert all(pattern_evaluations.values()), pattern_evaluations
{
    "framework": "LangGraph",
    "evaluation": pattern_evaluations,
    "native_abstractions": ["StateGraph", "Command", "Send", "ToolNode", "conditional edges"],
    "limitations": pattern_limitations,
}